# ディスク廃止：利益と信頼を同じモデルで比較する

全入力は**説明用の仮定**です。Sonyの顧客数・原価・行動反応を推定したものではありません。本文と同じPythonモジュールを使い、将来の行動を予言するのではなく損益分岐を確認します。金額は円です。

In [1]:
from pathlib import Path
import sys, json, copy
root = Path.cwd()
if not (root / "code" / "model.py").exists():
    raise RuntimeError("パッケージのルートを作業ディレクトリにしてください。")
sys.path.insert(0, str(root / "code"))
from model import simple_delta, exit_break_even, from_config, information_value
config = json.loads((root / "code/scenarios.json").read_text(encoding="utf-8"))
print(config["status"])

All inputs are illustrative assumptions. Not fitted to Sony data.


## 1. 追加離脱の損益分岐

同じ期間における従来の利益現在価値と、移行後の利益現在価値を比較します。増益には変動費削減を含め、固定費へ重ねません。

In [2]:
example = dict(n=1_000_000, baseline_ltv=40_000, uplift=3_000, fixed_pv=300_000_000, transition_cost=100_000_000)
for d in (0, .05, .08):
    print(f"追加離脱{d:.0%}: {simple_delta(**example, exit_share=d)/1e8:+.2f} 億円")
print(f"損益分岐: {exit_break_even(**example):.4%}")

追加離脱0%: +32.00 億円
追加離脱5%: +10.50 億円
追加離脱8%: -2.40 億円
損益分岐: 7.4419%


## 2. 五年のコホート比較

自然なデジタル移行は基準案にも起きます。顧客群、追加離脱、支出縮小、支援費用を分けます。支援による反応改善も仮定であり、投資すれば自動的に改善するという実証はありません。

In [3]:
results = from_config(config)
for name, result in results.items():
    print(f"{name:22s}: {result['npv']/1e8:+.3f} 億円")

baseline              : +0.000 億円
full_calm             : +12.495 億円
full_stress           : -10.151 億円
full_assurance        : +3.863 億円
selective_physical    : +2.125 億円


## 3. 自然移行の速度を変える

他の仮定を固定した一方向の感度分析です。将来の発生確率ではありません。

In [4]:
for rate in (0, .15, .30):
    altered = copy.deepcopy(config)
    altered["common"]["natural_digital_migration"] = rate
    npv = from_config(altered)["full_calm"]["npv"]
    print(f"自然移行{rate:.0%}: {npv/1e8:+.3f} 億円")

自然移行0%: +17.350 億円
自然移行15%: +12.495 億円
自然移行30%: +8.865 億円


## 4. 支援策が行動を変えなかった場合

費用だけが増える反証ケースを残します。

In [5]:
no_effect = copy.deepcopy(config)
no_effect["policies"][3]["responses"] = copy.deepcopy(no_effect["policies"][2]["responses"])
for r in no_effect["policies"][3]["responses"].values():
    r["annual_assurance_cost_per_member"] = 120
print(from_config(no_effect)["full_assurance"]["npv"] / 1e8, "億円")

-17.58030893062177 億円


## 5. 情報の価値の上限

確率55%は説明のために置いた重み。完全情報価値は現実の調査価値ではなく上限です。

In [6]:
information_value(good=4e9, bad=-3e9, good_probability=.55, research_cost=2e8)

{'adopt_without_information': 850000000.0000002,
 'best_without_information': 850000000.0000002,
 'perfect_information_value_before_cost': 2200000000.0,
 'evpi': 1349999999.9999998,
 'net_upper_bound_of_research': 1149999999.9999998}

## 自分の条件へ置き換える

`code/scenarios.json`を編集する前にコピーを保存してください。実データへ置き換える場合、対象期間・地域・コホート・利益の定義と入力の出所を記録します。将来の新規顧客や五年後以降はこの閉鎖コホートの外にあります。画像は `images/ps-disc-trust/`、モデルの範囲は `docs/MODEL_NOTES_JA.md` を参照してください。